In [1]:
# Load libraries

%load_ext autoreload
%autoreload 2

import pandas as pd
print("success: pandas")
import numpy as np
print("success: numpy")
import matplotlib.pyplot as plt
print("success: matplotlib")
import sklearn as sk
print("success: sklearn")
import seaborn as sns
print("success: c'est bon")
from pathlib import Path
dataset_train_filepath = Path('.') / 'datasets' / 'train.csv'
DATA_TRAIN_MAIN = pd.read_csv(dataset_train_filepath)
df_train = DATA_TRAIN_MAIN.copy()

success: pandas
success: numpy
success: matplotlib
success: sklearn
success: c'est bon


In [2]:
import mlsnips as mu
df_train.isna().sum().sort_values(ascending=False).head(19)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

In [3]:

# doing cleaning
# meaningful nulls
meaningful_nulls_cat = ['PoolQC','MiscFeature', 'Alley', 'Fence','FireplaceQu','MasVnrType']
meaningful_nulls_num = ['MasVnrArea']
error_nulls = ['LotFrontage']
df_train[meaningful_nulls_cat] = df_train[meaningful_nulls_cat].fillna('None')
df_train[meaningful_nulls_num] = df_train[meaningful_nulls_num].fillna(0) # quick patch
df_train['LotFrontage'] = df_train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median())) # oops

In [4]:
# average nulls
df_numerical = df_train.drop(columns=['Id']).select_dtypes(include=['float64', 'int64'])
df_numerical_columns = df_numerical.columns.to_list()
df_categorical = df_train.drop(columns=['Id']).select_dtypes(include=['object'])
df_categorical_columns = df_categorical.columns.to_list()
df_train.shape[1] - len(df_categorical_columns) - len(df_numerical_columns)

1

In [5]:
df_ordercat_maps = {
    "ExterQual":   {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "ExterCond":   {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "BsmtQual":    {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "BsmtCond":    {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "HeatingQC":   {"Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "KitchenQual": {"Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "FireplaceQu": {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "GarageQual":  {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
    "GarageCond":  {None:0, np.nan:0, "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5},
}

# for col in df_ordercat_maps.keys():
#     print(col)
#     print(df_train[col].unique())
#     print("============")

# ONLY DO THIS ONCE!!!!
# OR ELSE YOUR DATA WILL BE NAN-ED
for col, mapping in df_ordercat_maps.items():
    df_train[col] = df_train[col].map(mapping)

df_train[list(df_ordercat_maps.keys())].isna().sum()
# df_train[df_ordercat_maps.keys()].head()

ExterQual      0
ExterCond      0
BsmtQual       0
BsmtCond       0
HeatingQC      0
KitchenQual    0
FireplaceQu    0
GarageQual     0
GarageCond     0
dtype: int64

In [6]:
df_train = pd.get_dummies(df_train, drop_first=True)


In [7]:
# df_train.head()
df_categorical = df_train.drop(columns=['Id']).select_dtypes(include=['object'])
df_categorical_columns = df_categorical.columns.to_list()
df_train[df_categorical_columns].isna().sum().sum()

np.float64(0.0)

In [8]:
# cleaning up numerical average nulls
df_numerical_null_impute = {
    "LotFrontage": "median",
    "GarageYrBlt": "median",
    "MasVnrArea": "zero",
    "BsmtHalfBath": "zero",
    "BsmtFullBath": "zero",
    "GarageCars": "median",
    "GarageArea": "median",
    "BsmtFinSF2": "zero",
    "BsmtUnfSF": "median",
    "TotalBsmtSF": "median",
    "BsmtFinSF1": "median",
}
for col, method in df_numerical_null_impute.items():
    if method == "median":
        df_train[col] = df_train[col].fillna(df_train[col].median())
    elif method == "zero":
        df_train[col] = df_train[col].fillna(0)

df_train.head()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,60,65.0,8450,7,5,2003,2003,196.0,4,...,False,False,False,False,True,False,False,False,True,False
1,2,20,80.0,9600,6,8,1976,1976,0.0,3,...,False,False,False,False,True,False,False,False,True,False
2,3,60,68.0,11250,7,5,2001,2002,162.0,4,...,False,False,False,False,True,False,False,False,True,False
3,4,70,60.0,9550,7,5,1915,1970,0.0,3,...,False,False,False,False,True,False,False,False,False,False
4,5,60,84.0,14260,8,5,2000,2000,350.0,4,...,False,False,False,False,True,False,False,False,True,False


In [9]:
df_train.isna().sum().sort_values(ascending=False)


Id                       0
MSSubClass               0
LotFrontage              0
LotArea                  0
OverallQual              0
                        ..
SaleCondition_AdjLand    0
SaleCondition_Alloca     0
SaleCondition_Family     0
SaleCondition_Normal     0
SaleCondition_Partial    0
Length: 228, dtype: int64

In [10]:
X_baseline_1, y_baseline_1 = df_train.drop(columns=["Id","SalePrice"]), df_train["SalePrice"]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_baseline_1, y_baseline_1, test_size=0.2, random_state=42
)


In [11]:
# temporary comment
# mu.train_and_report_trees(X_train, X_val, y_train, y_val, report_models=True)

In [12]:

# "Try log-transforming the output"
X_baseline_2, y_baseline_2 = df_train.drop(columns=["Id","SalePrice"]), df_train["SalePrice"]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train_temp, y_val = train_test_split(
    X_baseline_2, y_baseline_2, test_size=0.2, random_state=42
)
# log-transform
y_train = np.log1p(y_train_temp)

# "Since outputs are log-transformed, error scores should also account for that."
# Shucks, have to take the whole function here then.
def train_and_report_trees_MODIFIED(X_train, X_val, y_train, y_val, report_models: bool=True):
    from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

    from sklearn.ensemble import (
        RandomForestRegressor,
        ExtraTreesRegressor,
        GradientBoostingRegressor
    )

    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor

    # Optional – enable only if installed / needed
    # from catboost import CatBoostRegressor

    tree_models = {
        "random-forest": RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),

        "extra-trees": ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),

        "gradient-boosting": GradientBoostingRegressor(
            random_state=42
        ),

        "xgboost": XGBRegressor(
            tree_method="hist",
            random_state=42,
            n_estimators=300,
            learning_rate=0.1,
            verbosity=0
        ),

        "light-gbm": LGBMRegressor(
            random_state=42,
            n_estimators=300,
            learning_rate=0.1
        ),

        # Enable only when categorical features are present
        # and CatBoost is installed
        # "catboost": CatBoostRegressor(
        #     random_state=42,
        #     verbose=False
        # ),
    }

    if report_models:
        print("Training tree models:")
        for name in tree_models:
            print(f" - {name}")

    records = []

    import time
    for name, model in tree_models.items():
        # ---- fit timing ----
        start_fit = time.time()
        model.fit(X_train, y_train)
        fit_time = time.time() - start_fit

        # ---- predict timing ----
        start_pred = time.time()
        preds = model.predict(X_val)
        # model predicts the LOG of the price.
        # we have to EXPONENTIATE back to get the real numbers.
        pred_time = time.time() - start_pred

        # METRICS EDITED HERE TO ACCOUNT FOR LOG-TRANSFORM!
        # ---- metrics ----
        mae = mean_absolute_error(y_val, np.expm1(preds))
        rmse = root_mean_squared_error(y_val, np.expm1(preds))
        r2 = r2_score(y_val, np.expm1(preds))

        records.append({
            "model": name,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            "fit_time_sec": fit_time,
            "pred_time_sec": pred_time
        })

    
    return pd.DataFrame(records).sort_values("mae")

# temporary comment
# train_and_report_trees_MODIFIED(X_train, X_val, y_train, y_val, report_models=True)

In [13]:
from xgboost import XGBRegressor
best_model_2 = XGBRegressor(tree_method="hist", random_state=42, n_estimators=300, learning_rate=0.1, verbosity=0)
best_model_2.fit(X_train, y_train)
y_pred_log = best_model_2.predict(X_val)
y_pred = np.expm1(y_pred_log)

best_model_2.get_booster().get_score(importance_type='gain')


{'MSSubClass': 0.0038766558282077312,
 'LotFrontage': 0.010027793236076832,
 'LotArea': 0.018203258514404297,
 'OverallQual': 2.00642991065979,
 'OverallCond': 0.042906370013952255,
 'YearBuilt': 0.06054787337779999,
 'YearRemodAdd': 0.03248882293701172,
 'MasVnrArea': 0.0035651109647005796,
 'ExterQual': 0.11061569303274155,
 'ExterCond': 0.005732086021453142,
 'BsmtQual': 0.19967688620090485,
 'BsmtCond': 0.05568934977054596,
 'BsmtFinSF1': 0.04480567201972008,
 'BsmtFinSF2': 0.006457892712205648,
 'BsmtUnfSF': 0.005569372326135635,
 'TotalBsmtSF': 0.09986994415521622,
 'HeatingQC': 0.010044525377452374,
 '1stFlrSF': 0.046044085174798965,
 '2ndFlrSF': 0.01987040415406227,
 'GrLivArea': 0.2290392369031906,
 'BsmtFullBath': 0.017109278589487076,
 'BsmtHalfBath': 0.022174330428242683,
 'FullBath': 0.07195695489645004,
 'HalfBath': 0.021174531430006027,
 'BedroomAbvGr': 0.024599337950348854,
 'KitchenAbvGr': 0.16343018412590027,
 'KitchenQual': 0.17254067957401276,
 'TotRmsAbvGrd': 0.014

In [14]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error


from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Optional – enable only if installed / needed
# from catboost import CatBoostRegressor

tree_models = {
    "random-forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "extra-trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "gradient-boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "xgboost": XGBRegressor(
        tree_method="hist",
        random_state=42,
        n_estimators=300,
        learning_rate=0.1,
        verbosity=0
    ),

    "light-gbm": LGBMRegressor(
        random_state=42,
        n_estimators=300,
        learning_rate=0.1
    ),

    # Enable only when categorical features are present
    # and CatBoost is installed
    # "catboost": CatBoostRegressor(
    #     random_state=42,
    #     verbose=False
    # ),
}

def kfold_rmse(model, X, y, n_splits=5, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        rmse = root_mean_squared_error(y_val, y_pred)
        rmses.append(rmse)

    return np.mean(rmses), np.std(rmses)

X_baseline_3, y_baseline_3 = df_train.drop(columns=["Id","SalePrice"]), df_train["SalePrice"]

# temporary comment
# for name, model in tree_models.items():
#     mean_rmse, std_rmse = kfold_rmse(model, X_baseline_3, y_baseline_3)
#     print(f"{name}: {mean_rmse:.0f} ± {std_rmse:.0f}")



In [15]:
def kfold_rmse_log_target(model, X, y, n_splits=5, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        y_train_log = np.log1p(y_train)

        model.fit(X_train, y_train_log)
        y_pred_log = model.predict(X_val)
        y_pred = np.expm1(y_pred_log)

        rmse = root_mean_squared_error(y_val, y_pred)
        rmses.append(rmse)

    return np.mean(rmses), np.std(rmses)

# temporary comment
# for name, model in tree_models.items():
#     mean_rmse, std_rmse = kfold_rmse_log_target(model, X_baseline_3, y_baseline_3)
#     print(f"{name}: {mean_rmse:.0f} ± {std_rmse:.0f}")


hyperparameters

In [16]:
from sklearn.model_selection import RandomizedSearchCV
xgb_model = XGBRegressor()
lightgbm_model = LGBMRegressor()

xgb_param_space = {
    "n_estimators": [300, 500, 800, 1200],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 1.5, 2]
}
lgbm_param_space = {
    "n_estimators": [500, 800, 1200, 2000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 5, 8, 10],
    "min_data_in_leaf": [10, 20, 40, 80],
    "feature_fraction": [0.6, 0.8, 1.0],
    "bagging_fraction": [0.6, 0.8, 1.0],
    "bagging_freq": [0, 1]
}

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_space,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

X_baseline_4, y_baseline_4 = df_train.drop(columns=["Id","SalePrice"]), df_train["SalePrice"]

# temporary comment
# search.fit(X_baseline_4, y_baseline_4)
# xgb, no log transform


In [17]:
# xgb_tunebest_nolog = search.best_estimator_
# -search.best_score_

In [18]:
search_2 = RandomizedSearchCV(
    estimator=lightgbm_model,
    param_distributions=lgbm_param_space,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# search_2.fit(X_baseline_4, y_baseline_4)
# # lightgbm, no log transform
# lgbm_tunebest_nolog = search_2.best_estimator_
# -search_2.best_score_

In [19]:
# results_1 = pd.DataFrame(search.cv_results_)
# results_1[ [ "mean_test_score", "std_test_score", "rank_test_score", "params" ] ].sort_values("rank_test_score").head(10)


In [20]:
# results_2 = pd.DataFrame(search_2.cv_results_)
# results_2[ [ "mean_test_score", "std_test_score", "rank_test_score", "params" ] ].sort_values("rank_test_score").head(10)

In [21]:
# results_1[["rank_test_score", "params"]].sort_values("rank_test_score").head(10)

In [22]:
# results_1 = results_1.sort_values("rank_test_score")
# top_trials_1 = results_1.head(10)
# pd.json_normalize(top_trials["params"].head(10))

xgb_param_space_2 = {
    "n_estimators": [350, 400, 450],
    "learning_rate": [0.035, 0.04, 0.045],
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 2, 3],
    "subsample": [0.5, 0.6, 0.7],
    "colsample_bytree": [0.5, 0.55, 0.6],
    "gamma": [0.1, 0.2, 0.3],
    "reg_alpha": [0.01, 0.05, 0.1],
    "reg_lambda": [1.2, 1.5, 1.8]
}

In [23]:
search_3 = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_space_2,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search_3.fit(X_baseline_4, y_baseline_4)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


,estimator,"XGBRegressor(...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.5, 0.55, ...], 'gamma': [0.1, 0.2, ...], 'learning_rate': [0.035, 0.04, ...], 'max_depth': [3, 4, ...], ...}"
,n_iter,40
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [24]:
results_3 = pd.DataFrame(search_3.cv_results_).sort_values("rank_test_score")
# results_3[ [ "mean_test_score", "std_test_score", "rank_test_score", "params" ] ].head(10)
top_trials_3 = results_3.head(10)
pd.json_normalize(top_trials_3["params"].head()).to_dict()



{'subsample': {0: 0.7, 1: 0.5, 2: 0.7, 3: 0.7, 4: 0.5},
 'reg_lambda': {0: 1.8, 1: 1.8, 2: 1.8, 3: 1.5, 4: 1.2},
 'reg_alpha': {0: 0.05, 1: 0.01, 2: 0.1, 3: 0.01, 4: 0.01},
 'n_estimators': {0: 400, 1: 450, 2: 450, 3: 350, 4: 350},
 'min_child_weight': {0: 1, 1: 1, 2: 2, 3: 2, 4: 1},
 'max_depth': {0: 4, 1: 4, 2: 5, 3: 5, 4: 5},
 'learning_rate': {0: 0.04, 1: 0.045, 2: 0.045, 3: 0.04, 4: 0.035},
 'gamma': {0: 0.3, 1: 0.1, 2: 0.1, 3: 0.1, 4: 0.2},
 'colsample_bytree': {0: 0.5, 1: 0.55, 2: 0.55, 3: 0.6, 4: 0.6}}

In [25]:
# Get top 5 configs by rank
top5_idx = search_3.cv_results_['rank_test_score'].argsort()[:5]
top5_params = [search_3.cv_results_['params'][i] for i in top5_idx]

X_baseline_4, y_baseline_4 = df_train.drop(columns=["Id","SalePrice"]), df_train["SalePrice"]

X_train, X_val, y_train, y_val = train_test_split(
    X_baseline_4, y_baseline_4, test_size=0.2, random_state=42
)

full_models = []
for params in top5_params:
    model = XGBRegressor(**params, random_state=42)
    model.fit(X_train, y_train)  # use all training data
    full_models.append(model)

top_model = XGBRegressor(**top5_params[0], random_state=42)
top_model.fit(X_train, y_train)

# Ensemble predictions
preds_ensemble = np.column_stack([m.predict(X_val) for m in full_models])
ensemble_pred = preds_ensemble.mean(axis=1)

# Single top model predictions
top_pred = top_model.predict(X_val)

# Compare RMSE
rmse_ensemble = root_mean_squared_error(y_val, ensemble_pred)
rmse_top = root_mean_squared_error(y_val, top_pred)

print(f"Ensemble RMSE: {rmse_ensemble:.2f}")
print(f"Top single model RMSE: {rmse_top:.2f}")


Ensemble RMSE: 23608.31
Top single model RMSE: 23199.90


In [27]:
# from sklearn.base import BaseEstimator, RegressorMixin
# class XGBEnsemble(BaseEstimator, RegressorMixin):
#     def __init__(self, params_list):
#         """
#         params_list: list of dicts, each dict is a hyperparameter set for one XGBRegressor
#         """
#         self.params_list = params_list
#         self.models = []

#     def fit(self, X, y):
#         self.models = []
#         for params in self.params_list:
#             model = XGBRegressor(**params, random_state=42)
#             model.fit(X, y)
#             self.models.append(model)
#         return self

#     def predict(self, X):
#         if not self.models:
#             raise RuntimeError("You must fit the ensemble before predicting")
#         preds = np.column_stack([m.predict(X) for m in self.models])
#         return preds.mean(axis=1)  # simple averaging

from sklearn.pipeline import Pipeline
from ames_preprocessor import AmesPreprocessor
from xgbtrained import XGBEnsemble

# Assume AmesPreprocessor() is defined as in the previous step
preprocessor = AmesPreprocessor()

ensemble_model = XGBEnsemble(top5_params)  # top 5 configs from search_3

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('ensemble', ensemble_model)
])

# Train on full training data
df_train_tester = DATA_TRAIN_MAIN.copy()
# df_train_tester
X_final, y_final = df_train_tester.drop(columns=["Id","SalePrice"]), df_train_tester["SalePrice"]
pipeline.fit(X_final, y_final)
pipeline
# import joblib

# joblib.dump(pipeline, 'ames_xgb_ensemble_pipeline.pkl')

,steps,"[('preprocessing', ...), ('ensemble', ...)]"
,transform_input,None
,memory,None
,verbose,False
,params_list,"[{'colsample_bytree': 0.5, 'gamma': 0.3, 'learning_rate': 0.04, 'max_depth': 4, ...}, {'colsample_bytree': 0.55, 'gamma': 0.1, 'learning_rate': 0.045, 'max_depth': 4, ...}, ...]"


In [29]:
dataset_test_filepath = Path('.') / 'datasets' / 'test.csv'
DATA_TEST_MAIN = pd.read_csv(dataset_test_filepath)
df_test = DATA_TEST_MAIN.copy()
X_problem = df_test.drop(columns=["Id"])
X_problem


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


In [31]:
y_submission = pipeline.predict(X_problem)
y_submission

c:\Users\User\Documents\Code\python\ml\.venv\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


array([127406.164, 159685.45 , 183814.7  , ..., 166713.78 , 116900.2  ,
       220448.2  ], shape=(1459,), dtype=float32)

In [32]:
submission = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": y_submission
})

submission.head()


,Id,SalePrice
0,1461,127406.164062
1,1462,159685.453125
2,1463,183814.703125
3,1464,189155.656250
4,1465,185138.062500


In [33]:
assert len(submission) == len(df_test)
assert submission["Id"].is_unique
assert not submission["SalePrice"].isna().any()


In [34]:
submission.to_csv("submission.csv", index=False)
